<a href="https://colab.research.google.com/github/singh-deepak7/natural-language-processing/blob/main/NMT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Translation

In [1]:
# import necessary libraries

# Preprocessing Libraries

import torch
import pandas as pd
import nltk
import spacy

# Evaluation Libraries

from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

# Transformer Libraries for MarianMt
from transformers import MarianMTModel, MarianTokenizer

In [2]:
# Initialize The MarianMt Model

model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)

model = MarianMTModel.from_pretrained(model_name)
#

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [3]:
# Translate using pre-trained model

def translate(text):
  inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
  outputs = model.generate(**inputs)
  return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [5]:
text = 'How are you'
translate(text)

'Comment allez-vous ?'

In [6]:
# Load CSV file
df = pd.read_csv('/content/opus_books_en-fr_test.csv')
df.head()

,en,fr
0,She looked out the window.,Elle regarda par la fenêtre.
1,He opened the book and started reading.,Il ouvrit le livre et commença à lire.
2,The sun was shining brightly.,Le soleil brillait de mille feux.
3,They walked along the river.,Ils marchaient le long de la rivière.
4,It was a quiet afternoon.,C'était un après-midi tranquille.


In [7]:
preds = [ translate(en) for en in df['en']]
preds

['Elle a regardé par la fenêtre.',
 'Il a ouvert le livre et a commencé à lire.',
 'Le soleil brillait brillamment.',
 'Ils marchaient le long de la rivière.',
 "C'était un après-midi tranquille.",
 'Elle portait une écharpe rouge.',
 'Il lui chuchotait un secret.',
 'Les enfants riaient et jouaient.',
 'La chambre sentait des fleurs fraîches.',
 'Elle tourna la page lentement.']

In [8]:
# Evaluation Metrics using Semantic Similarity

actual = list(df['fr'])
actual



['Elle regarda par la fenêtre.',
 'Il ouvrit le livre et commença à lire.',
 'Le soleil brillait de mille feux.',
 'Ils marchaient le long de la rivière.',
 "C'était un après-midi tranquille.",
 'Elle portait une écharpe rouge.',
 'Il lui chuchota un secret.',
 'Les enfants riaient et jouaient.',
 'La pièce sentait les fleurs fraîches.',
 'Elle tourna la page lentement.']

In [9]:
semantic_model = SentenceTransformer("sentence-transformers/LaBSE")

modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

In [11]:
!python -m spacy download fr_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 59.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [12]:
## To generate tokenized input for Sentence Transformer for French Text

nlp_fr = spacy.load("fr_core_news_sm")

In [13]:
def normalize_fr(text):
  doc = nlp_fr(text)
  return " ".join([token.lemma_ for token in doc])

In [14]:
normalize_fr('Elle regarda par la fenêtre.')

'lui regarder par le fenêtre .'

In [15]:
def check_semantic(ref, pred):
  # Normalize actual text
  ref_norm = normalize_fr(ref)

  # Normalize our Predicted Text
  pred_norm = normalize_fr(pred)

  # Generate embeddings for both normalized actual text and predicted text
  embs = semantic_model.encode([ref_norm, pred_norm])

  # Check the cosine similary between both of the embeddings
  return util.cos_sim(embs[0], embs[1]).item()

In [20]:
actual[2]

'Le soleil brillait de mille feux.'

In [21]:
preds[2]

'Le soleil brillait brillamment.'

In [18]:
check_semantic(actual[0], preds[0])

0.9521220922470093

In [19]:
for i in range(len(df['en'])):
  ref = df['fr'].iloc[i]
  pred = preds[i]

  print('\n Similarity Score: ------> ', check_semantic(ref, pred))


 Similarity Score: ------>  0.9521220922470093

 Similarity Score: ------>  0.9595828056335449

 Similarity Score: ------>  0.8569494485855103

 Similarity Score: ------>  1.0

 Similarity Score: ------>  1.0

 Similarity Score: ------>  1.0000001192092896

 Similarity Score: ------>  1.000000238418579

 Similarity Score: ------>  1.0000001192092896

 Similarity Score: ------>  0.9203620553016663

 Similarity Score: ------>  1.0
